# Notebook 5: Advanced Python & Beyond

## ⚠️ Important Note

**This notebook goes BEYOND what you need to build and understand the SWBM.**

Notebooks 1-4 contain everything you need to:
- Understand how Python works
- Load and manipulate data
- Write functions and loops
- Build your own environmental model

**This notebook is for students who want to take their Python skills further** - for future projects, research, or a career in data science. It's entirely optional.

## Learning Goals
- Write faster, more efficient code
- Debug and test your code like a professional
- Create publication-quality visualizations
- Organize code with object-oriented programming
- Discover what else Python can do

In [ ]:
import numpy as np
import pandas as pd
import time

# Example: Calculate ET fraction for 10,000 values
num_values = 10000

soil_moistures = np.random.uniform(0, 200, num_values)
c_s = 200
b0 = 0.7
g = 1.5

# Method 1: Loop (SLOW)
start_time = time.time()
et_fracs_slow = []
for sm in soil_moistures:
    sm_capped = min(sm, c_s)
    et_frac = b0 * (sm_capped / c_s) ** g
    et_fracs_slow.append(et_frac)
time_slow = time.time() - start_time
et_fracs_slow = np.array(et_fracs_slow)

# Method 2: NumPy vectorization (FAST)
start_time = time.time()
soil_moistures_capped = np.minimum(soil_moistures, c_s)
et_fracs_fast = b0 * (soil_moistures_capped / c_s) ** g
time_fast = time.time() - start_time

print(f"Loop method: {time_slow:.4f} seconds")
print(f"NumPy method: {time_fast:.4f} seconds")
print(f"Speedup: {time_slow / time_fast:.1f}x faster with NumPy!")
print(f"\nResults are identical: {np.allclose(et_fracs_slow, et_fracs_fast)}")

### Key Optimization Rules:
1. **Use NumPy** instead of loops for numerical data
2. **Avoid appending** to lists in loops (create arrays upfront)
3. **Preallocate** arrays when you know the size
4. **Use vectorized operations** (.sum(), .mean(), etc.)

In [ ]:
# GOOD: Preallocate and vectorize
def predict_ts_vectorized(data, config):
    """
    Optimized version using full vectorization.
    """
    n_days = data.shape[0]
    
    # Preallocate arrays
    moists = np.zeros(n_days)
    runoff_fracs = np.zeros(n_days)
    et_fracs = np.zeros(n_days)
    
    c_s = config['c_s']
    b0 = config['b0']
    g = config['g']
    a = config['a']
    
    moists[0] = 0.9 * c_s
    
    precip = np.asarray(data['tp'])
    snr = np.asarray(data['snr'])
    
    for i in range(n_days):
        # Vectorized operations
        sm_capped = min(moists[i], c_s)
        et_fracs[i] = b0 * (sm_capped / c_s) ** g
        runoff_fracs[i] = min((moists[i] / c_s) ** a, 1.0)
        
        if i < n_days - 1:
            moists[i + 1] = moists[i] + (precip[i] - et_fracs[i] * snr[i] - runoff_fracs[i] * precip[i])
            moists[i + 1] = max(moists[i + 1], 0)  # No negative SM
    
    # Vectorized conversions
    runoffs = runoff_fracs * precip
    ets = et_fracs * snr
    
    return moists, runoffs, ets

print("Optimized function created. This approach is much faster for large datasets!")

## 2. Debugging: Finding and Fixing Errors

Debugging techniques to find problems in your code:

In [ ]:
# Technique 1: Print debugging - add print statements
def buggy_water_balance(precip, et, runoff):
    """Calculate water balance change (has a bug!)."""
    # Add debugging prints
    print(f"DEBUG: precip={precip}, et={et}, runoff={runoff}")
    
    balance = precip - et  # BUG: forgot to subtract runoff!
    print(f"DEBUG: balance={balance}")
    return balance

result = buggy_water_balance(50, 20, 10)
print(f"Result should be 20 (50-20-10), but got: {result}")
print("The debug output helps us see what's happening!")

In [ ]:
# Technique 2: Assertions - check assumptions
def water_balance_safe(precip, et, runoff):
    """Calculate water balance with safety checks."""
    # Check inputs
    assert precip >= 0, "Precipitation must be non-negative"
    assert et >= 0, "ET must be non-negative"
    assert runoff >= 0, "Runoff must be non-negative"
    assert precip > et or et > 0, "ET can't exceed precipitation unless soil provides water"
    
    balance = precip - et - runoff
    return balance

# This works
print(f"Normal case: {water_balance_safe(50, 20, 10)}")

# This fails with a clear error message
try:
    water_balance_safe(-5, 20, 10)  # Negative precipitation
except AssertionError as e:
    print(f"Caught error: {e}")

In [ ]:
# Technique 3: Try-except for handling errors gracefully
def load_data_safely(filename):
    """Load data and handle errors gracefully."""
    try:
        data = pd.read_csv(filename)
        print(f"Successfully loaded {filename}")
        return data
    except FileNotFoundError:
        print(f"Error: File {filename} not found")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

# This will handle errors nicely
data = load_data_safely('data/Data_swbm_Germany.csv')
data_missing = load_data_safely('nonexistent_file.csv')

## 5. Object-Oriented Programming - Organizing Complex Code

For larger projects, organize code using classes:

In [ ]:
# Define a class to manage environmental data
class EnvironmentalDataset:
    """A class to manage and analyze environmental data."""
    
    def __init__(self, name, filepath=None):
        """Initialize the dataset."""
        self.name = name
        self.data = None
        self.filepath = filepath
        if filepath:
            self.load_data()
    
    def load_data(self, filepath=None):
        """Load data from CSV file."""
        if filepath:
            self.filepath = filepath
        
        if not self.filepath:
            raise ValueError("No filepath specified")
        
        try:
            self.data = pd.read_csv(self.filepath)
            print(f"✓ Loaded {len(self.data)} records for {self.name}")
        except FileNotFoundError:
            print(f"✗ File not found: {self.filepath}")
            self.data = None
    
    def get_summary_stats(self):
        """Calculate summary statistics for all columns."""
        if self.data is None:
            print("No data loaded")
            return None
        
        return self.data.describe()
    
    def plot_time_series(self, column):
        """Plot a time series of a specific column."""
        if self.data is None:
            print("No data loaded")
            return
        
        if column not in self.data.columns:
            print(f"Column '{column}' not found")
            return
        
        plt.figure(figsize=(12, 5))
        plt.plot(self.data[column], linewidth=1.5)
        plt.xlabel('Time')
        plt.ylabel(column)
        plt.title(f'{self.name} - {column}')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def get_monthly_aggregates(self, column):
        """Calculate monthly averages for a column."""
        if self.data is None:
            print("No data loaded")
            return None
        
        if 'time' not in self.data.columns:
            print("No 'time' column found for aggregation")
            return None
        
        monthly = self.data.groupby(pd.Grouper(key='time', freq='M'))[column].mean()
        return monthly

# Use the class
dataset = EnvironmentalDataset('Germany Dataset', 'data/Data_swbm_Germany.csv')
print(f"\nDataset Info: {dataset.name}")
print(f"Columns: {dataset.data.columns.tolist()}")
print(f"\nStatistics for precipitation:")
print(dataset.data['tp_[mm]'].describe())

## 6. Statistical Analysis - Going Deeper

Use SciPy for advanced statistics:

In [ ]:
from scipy.stats import pearsonr, spearmanr, gaussian_kde
from scipy.optimize import minimize

# Load data
data = pd.read_csv('data/Data_swbm_Germany.csv')

# Calculate different correlations
precip = data['tp_[mm]']
sm = data['sm_[m3/m3]'] * 1000

# Pearson correlation (assumes linear relationship)
pearson_r, pearson_p = pearsonr(precip, sm)
print(f"Pearson Correlation: {pearson_r:.3f} (p={pearson_p:.2e})")
print(f"  → Linear relationship strength")

# Spearman correlation (rank-based, more robust)
spearman_r, spearman_p = spearmanr(precip, sm)
print(f"\nSpearman Correlation: {spearman_r:.3f} (p={spearman_p:.2e})")
print(f"  → Monotonic relationship (more robust to outliers)")

# Kernel Density Estimation for probability distributions
kde_precip = gaussian_kde(precip)
x_range = np.linspace(0, precip.max(), 200)
pdf = kde_precip(x_range)

plt.figure(figsize=(10, 5))
plt.hist(precip, bins=30, density=True, alpha=0.5, label='Histogram')
plt.plot(x_range, pdf, 'r-', linewidth=2, label='KDE (Probability Distribution)')
plt.xlabel('Daily Precipitation (mm)')
plt.ylabel('Probability Density')
plt.title('Distribution Estimation Using Kernel Density Estimation')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Optimization - Finding Best Parameter Values

Use optimization when you need to find parameter values that minimize error:

In [ ]:
from scipy.optimize import minimize

# Simple example: fit a parabola to data
# Create sample data: y = ax² + bx + c + noise
x_data = np.linspace(-5, 5, 50)
y_true = 2 * x_data**2 - 3 * x_data + 1
y_observed = y_true + np.random.normal(0, 2, len(x_data))  # Add noise

def calculate_error(params):
    """Calculate sum of squared errors between model and observations."""
    a, b, c = params
    y_model = a * x_data**2 + b * x_data + c
    error = np.sum((y_observed - y_model) ** 2)
    return error

# Initial guess
initial_params = [1, 0, 0]

# Optimize
result = minimize(calculate_error, initial_params, method='Nelder-Mead')

print("Parameter Optimization Results:")
print(f"Optimal parameters:")
print(f"  a (quadratic coefficient): {result.x[0]:.3f} (true: 2.0)")
print(f"  b (linear coefficient): {result.x[1]:.3f} (true: -3.0)")
print(f"  c (constant): {result.x[2]:.3f} (true: 1.0)")
print(f"\nFinal error: {result.fun:.2f}")
print(f"Optimization successful: {result.success}")

# Plot results
plt.figure(figsize=(10, 5))
plt.scatter(x_data, y_observed, alpha=0.5, s=50, label='Observed data')
plt.plot(x_data, y_true, 'g-', linewidth=2, label='True function')

y_fitted = result.x[0] * x_data**2 + result.x[1] * x_data + result.x[2]
plt.plot(x_data, y_fitted, 'r--', linewidth=2, label='Fitted function')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Parameter Optimization: Fitting a Parabola to Noisy Data')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Your Learning Path Forward

Here's how you can continue building your Python skills:

In [ ]:
# Create a learning roadmap
roadmap = {
    'Beginner (Notebooks 1-4)': [
        '✓ Variables and data types',
        '✓ Lists and dictionaries',
        '✓ Functions and loops',
        '✓ Control flow (if/else)',
        '✓ Pandas DataFrames',
        '✓ NumPy arrays',
        '✓ Basic plotting',
        '✓ Building models'
    ],
    'Intermediate (This Notebook)': [
        '• Code optimization',
        '• Debugging and testing',
        '• Professional visualization',
        '• Object-oriented programming',
        '• Statistical analysis'
    ],
    'Advanced (Next Steps)': [
        'Machine learning with scikit-learn',
        'Geospatial analysis (geopandas, rasterio)',
        'Working with large datasets (Dask)',
        'Climate data (xarray, netCDF)',
        'Parallel computing',
        'Creating Python packages',
        'Contributing to open source'
    ]
}

print("Your Python Learning Roadmap\n" + "="*60)
for level, topics in roadmap.items():
    print(f"\n{level}:")
    for topic in topics:
        print(f"  {topic}")

## 9. Recommended Learning Resources

Free online resources to continue learning:

In [ ]:
resources = {
    'Official Documentation': [
        'https://docs.python.org/',
        'https://numpy.org/doc/',
        'https://pandas.pydata.org/docs/',
        'https://matplotlib.org/'
    ],
    'Tutorials & Learning': [
        'https://realpython.com/',
        'https://www.codecademy.com/learn/learn-python',
        'https://earthpy.readthedocs.io/',
        'https://xarray.readthedocs.io/'
    ],
    'Practice & Challenges': [
        'https://kaggle.com/',
        'https://leetcode.com/',
        'https://github.com/ (find open source projects)'
    ],
    'Communities': [
        'Stack Overflow',
        'Reddit: r/learnprogramming, r/Python',
        'GitHub Discussions',
        'Local Python meetups'
    ]
}

print("Recommended Learning Resources\n" + "="*60)
for category, sources in resources.items():
    print(f"\n{category}:")
    for source in sources:
        print(f"  • {source}")

## 10. Project Ideas - Putting Skills to Use

Ideas for your next projects:

In [ ]:
projects = {
    'Easy (Using Notebooks 1-4)': [
        'Analyze your own environmental data using Notebooks 1-2 techniques',
        'Create a more complex environmental model based on Notebook 4',
        'Compare different datasets using data manipulation skills'
    ],
    'Medium (Using This Notebook)': [
        'Optimize your model using vectorization techniques',
        'Add comprehensive error checking and testing',
        'Create publication-quality figures',
        'Find optimal parameters for a model using optimization'
    ],
    'Hard (Next Level)': [
        'Build an interactive web app (Jupyter Voila or Dash)',
        'Use machine learning to predict environmental outcomes',
        'Analyze geospatial climate data (rasterio, GeoPandas)',
        'Create a Python package for your models',
        'Contribute to open source environmental science projects'
    ]
}

print("Project Ideas - Apply Your Skills\n" + "="*60)
for difficulty, ideas in projects.items():
    print(f"\n{difficulty}:")
    for idea in ideas:
        print(f"  • {idea}")

## Exercises

### Exercise 1: Optimize a Calculation
Take any loop-based calculation from previous notebooks and rewrite it using NumPy vectorization. Measure the speedup.

In [ ]:
# YOUR CODE HERE
# import time
# # Loop version
# start = time.time()
# # ... loop code ...
# time_loop = time.time() - start
# 
# # Vectorized version
# start = time.time()
# # ... vectorized code ...
# time_vectorized = time.time() - start
# 
# print(f"Speedup: {time_loop / time_vectorized:.1f}x")

### Exercise 2: Write Tests
Take a function from your model and write 3-4 unit tests to verify it works correctly in different scenarios.

In [ ]:
# YOUR CODE HERE
# def my_function(x, y):
#     return x + y
# 
# def test_my_function():
#     assert my_function(2, 3) == 5
#     assert my_function(0, 0) == 0
#     assert my_function(-1, 1) == 0
#     print("All tests passed!")
# 
# test_my_function()

### Exercise 3: Create a Class
Build your own class for analyzing environmental data with at least 3 methods:
- Load data
- Calculate statistics
- Create a visualization

In [ ]:
# YOUR CODE HERE
# class MyDataset:
#     def __init__(self, name):
#         self.name = name
#         self.data = None
#     
#     def load_data(self, filepath):
#         self.data = pd.read_csv(filepath)
#     
#     def stats(self):
#         return self.data.describe()
#     
#     def plot(self, column):
#         plt.plot(self.data[column])
#         plt.show()

### Exercise 2: Optimize Your Model Code
Take any loop-based calculation from previous notebooks and rewrite it using NumPy vectorization. Measure the speedup.

In [ ]:
# YOUR CODE HERE
# import time
# # Loop version
# start = time.time()
# # ... loop code ...
# time_loop = time.time() - start
# 
# # Vectorized version
# start = time.time()
# # ... vectorized code ...
# time_vectorized = time.time() - start
# 
# print(f"Speedup: {time_loop / time_vectorized:.1f}x")

### Exercise 3: Add Error Handling
Take one of your functions from earlier notebooks and add:
- Input validation with assertions
- Try-except blocks for error handling
- Debug print statements

In [ ]:
# YOUR CODE HERE
# def robust_calculation(precip, et, runoff):
#     """Calculate with error checking."""
#     try:
#         assert isinstance(precip, (int, float)), "precip must be a number"
#         assert precip >= 0, "precip must be non-negative"
#         # ... more assertions ...
#         
#         result = precip - et - runoff
#         return result
#     except AssertionError as e:
#         print(f"Error: {e}")
#         return None
#     except Exception as e:
#         print(f"Unexpected error: {e}")
#         return None

## Final Thoughts

### What You've Accomplished
Through all 5 notebooks, you've learned:
- ✓ **Foundation:** Python basics, data types, control flow
- ✓ **Data Handling:** Loading, manipulating, analyzing real datasets
- ✓ **Modeling:** Building environmental models from scratch
- ✓ **Advanced Skills:** Optimization, testing, visualization, OOP

### Why This Matters
These skills enable you to:
- Work with real environmental and climate data
- Build, understand, and improve scientific models
- Analyze large datasets efficiently
- Collaborate on research projects
- Pursue a career in data science, environmental science, or climate research

### Continue Learning
The best way to master Python:
1. **Practice** - Write code every day
2. **Build** - Create projects that interest you
3. **Read** - Study other people's code
4. **Share** - Contribute to open source or help peers
5. **Ask** - Don't hesitate to seek help

### Remember
- Programming is a skill, not a talent - anyone can learn it
- The best programmers are good at Googling problems
- Every expert was once a beginner
- Your code doesn't need to be perfect, it needs to work
- There's always more to learn (and that's the fun part!)

Good luck on your Python journey! 🚀